In [15]:
import asyncio
import threading
import json
import time
import os
import cv2
import numpy as np
from matplotlib import pyplot as plt
import sklearn.cluster
import skimage
from skimage import morphology
import argparse
import matplotlib.cm as cm


In [ ]:
def whitebalance(image):
    result = image.astype(np.float32)
    avg_r = np.mean(result[:, :, 0])
    avg_g = np.mean(result[:, :, 1])
    avg_b = np.mean(result[:, :, 2])
    avg_gray = (avg_r + avg_g + avg_b) / 3.0

    scale_r = avg_gray / avg_r if avg_r > 0 else 1.0
    scale_g = avg_gray / avg_g if avg_g > 0 else 1.0
    scale_b = avg_gray / avg_b if avg_b > 0 else 1.0

    result[:, :, 0] *= scale_r
    result[:, :, 1] *= scale_g
    result[:, :, 2] *= scale_b

    result = np.clip(result, 0, 255)
    return result.astype(np.uint8)
    

In [ ]:
image = cv2.imread('image_2025-11-03_11-15-13.png', cv2.IMREAD_COLOR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Create comparison of all methods
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Original image
axes[0, 0].imshow(image)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

# Gray World method
balanced_gw = whitebalance(image, method='grayworld')
axes[0, 1].imshow(balanced_gw)
axes[0, 1].set_title('Gray World')
axes[0, 1].axis('off')

# Max White method
balanced_mw = whitebalance(image, method='max_white')
axes[0, 2].imshow(balanced_mw)
axes[0, 2].set_title('Max White')
axes[0, 2].axis('off')

# Retinex method
balanced_ret = whitebalance(image, method='retinex')
axes[1, 0].imshow(balanced_ret)
axes[1, 0].set_title('Retinex')
axes[1, 0].axis('off')

# Reference-based method (most common color)
print("\n--- Reference-based method ---")
balanced_ref = whitebalance_reference(image, target=200)
axes[1, 1].imshow(balanced_ref)
axes[1, 1].set_title('Reference (Most Common Color)')
axes[1, 1].axis('off')

# Empty subplot
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def whitebalance(image, method='grayworld'):
    """
    White balance correction with multiple methods.
    
    Args:
        image: RGB image (numpy array)
        method: 'grayworld', 'max_white', or 'retinex'
    
    Returns:
        White balanced RGB image
    """
    result = image.astype(np.float32)
    
    if method == 'grayworld':
        # Gray World assumption - average of each channel should be gray
        avg_r = np.mean(result[:, :, 0])
        avg_g = np.mean(result[:, :, 1])
        avg_b = np.mean(result[:, :, 2])
        
        avg_gray = (avg_r + avg_g + avg_b) / 3.0
        
        # Scale factors
        scale_r = avg_gray / avg_r if avg_r > 0 else 1.0
        scale_g = avg_gray / avg_g if avg_g > 0 else 1.0
        scale_b = avg_gray / avg_b if avg_b > 0 else 1.0
        
        result[:, :, 0] = result[:, :, 0] * scale_r
        result[:, :, 1] = result[:, :, 1] * scale_g
        result[:, :, 2] = result[:, :, 2] * scale_b
        
    elif method == 'max_white':
        # Max White assumption - brightest areas should be white
        max_r = np.percentile(result[:, :, 0], 98)
        max_g = np.percentile(result[:, :, 1], 98)
        max_b = np.percentile(result[:, :, 2], 98)
        
        result[:, :, 0] = result[:, :, 0] * (255.0 / max_r) if max_r > 0 else result[:, :, 0]
        result[:, :, 1] = result[:, :, 1] * (255.0 / max_g) if max_g > 0 else result[:, :, 1]
        result[:, :, 2] = result[:, :, 2] * (255.0 / max_b) if max_b > 0 else result[:, :, 2]
        
    elif method == 'retinex':
        # Simplified Retinex - removes color cast by normalizing each channel
        for i in range(3):
            channel = result[:, :, i]
            channel_blur = cv2.GaussianBlur(channel, (0, 0), sigmaX=50)
            # Avoid division by zero
            channel_blur = np.maximum(channel_blur, 1.0)
            result[:, :, i] = (channel / channel_blur) * 128
    
    # Clip values to valid range
    result = np.clip(result, 0, 255)
    
    return result.astype(np.uint8)


def whitebalance_reference(image, target=200):
    result = image.astype(np.float32)
    quantized = (image // 16) * 16  # Reduce to 16 levels per channel
    pixels = quantized.reshape(-1, 3)
    
    # Find the most common color
    unique_colors, counts = np.unique(pixels, axis=0, return_counts=True)
    most_common_idx = np.argmax(counts)
    reference_color = unique_colors[most_common_idx].astype(np.float32)
    
    print(f"Most common color (background): RGB({reference_color[0]:.0f}, {reference_color[1]:.0f}, {reference_color[2]:.0f})")
    
    # Calculate scaling factors to make this color neutral
    scale_r = target / reference_color[0] if reference_color[0] > 0 else 1.0
    scale_g = target / reference_color[1] if reference_color[1] > 0 else 1.0
    scale_b = target / reference_color[2] if reference_color[2] > 0 else 1.0
    
    print(f"Scale factors: R={scale_r:.3f}, G={scale_g:.3f}, B={scale_b:.3f}")
    
    result[:, :, 0] = result[:, :, 0] * scale_r
    result[:, :, 1] = result[:, :, 1] * scale_g
    result[:, :, 2] = result[:, :, 2] * scale_b
    result = np.clip(result, 0, 255)
    
    return result.astype(np.uint8)